### ============================================
# Parkinson's Disease vs Control EEG Classifier
# Full Pipeline: BIDS loading, Preprocessing, ICA, Feature Extraction, ML
# Dataset: ds002778 (BIDS format)
# ============================================

import os
import os.path as op
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals
from mne.preprocessing import ICA, create_eog_epochs
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import os
import openneuro

# Define BIDS root folder
bids_root = "PD1"
os.makedirs(bids_root, exist_ok=True)

# Download dataset (all subjects, all sessions)
openneuro.download(dataset="ds002778", target_dir=bids_root)

print("✅ Download complete.")


In [ ]:
import os

subjects = [d for d in os.listdir(bids_root) if d.startswith("sub-")]
print(f"👥 Subjects found ({len(subjects)} total):")
for subject in subjects:
    print(f" - {subject}")


In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load participant-level data
df = pd.read_csv('PD1/participants.tsv', sep='\t')

# Encode categorical variables
le_gender = LabelEncoder()
df['gender'] = le_gender.fit_transform(df['gender'])

le_hand = LabelEncoder()
df['hand'] = le_hand.fit_transform(df['hand'])

# Initialize
features = []
labels = []
subject_ids = []

bids_root = "PD1"  # top-level BIDS directory

# Walk through all subject/session folders
for subject in os.listdir(bids_root):
    subj_path = os.path.join(bids_root, subject)
    if not os.path.isdir(subj_path) or not subject.startswith("sub-"):
        continue

    subj_row = df[df["participant_id"] == subject]
    if subj_row.empty:
        print(f"Skipping {subject}: not in participants.tsv")
        continue
    subj_row = subj_row.iloc[0]

    # Extract demographic/clinical data
    age = subj_row["age"]
    gender = subj_row["gender"]
    hand = subj_row["hand"]
    mmse = subj_row["MMSE"]
    naart = subj_row["NAART"]
    disease_duration = subj_row["disease_duration"] if not pd.isna(subj_row["disease_duration"]) else 0

    # Loop through sessions (ses-on, ses-off, ses-hc)
    for session in os.listdir(subj_path):
        if not session.startswith("ses-"):
            continue

        session_path = os.path.join(subj_path, session)
        if not os.path.isdir(session_path):
            continue

        # Determine label from session type
        if session == "ses-hc":
            label = 0  # Control
        elif session == "ses-on":
            label = 1  # PD-On
        elif session == "ses-off":
            label = 2  # PD-Off
        else:
            print(f"Unknown session {session} for {subject}, skipping.")
            continue

        # Append feature vector and label
        features.append(np.array([age, gender, hand, mmse, naart, disease_duration]))
        labels.append(label)
        subject_ids.append(f"{subject}_{session}")

# Convert to tensors
features_tensor = torch.tensor(features, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long)

# Print basic info
print(f"Features tensor shape: {features_tensor.shape}")
print(f"Labels tensor shape: {labels_tensor.shape}")
print(f"Label counts: {np.bincount(labels_tensor.numpy())}")


## Step 1: Define Feature Extraction Function

## Step 2: Preprocess and Extract for Subject

In [ ]:
import time
import psutil
import numpy as np
import mne
import os
from mne.preprocessing import ICA, create_eog_epochs
from mne_bids import BIDSPath, read_raw_bids

# --- Feature extraction function (assumed to exist) ---
# from your_module import extract_band_power_epochs

# Categorize session types
def categorize_session(session_name):
    if 'ses-on' in session_name:
        return 'pd-on'
    elif 'ses-off' in session_name:
        return 'pd-off'
    elif 'ses-hc' in session_name:
        return 'ctl'
    else:
        return 'unknown'

# Preprocess and extract features
def preprocess_and_extract_features(raw_data):
    print("Starting preprocessing and feature extraction...")

    # 1. Create synthetic events (if no annotations)
    print("Creating synthetic events...")
    if len(raw_data.annotations) == 0:
        events = np.array([
            [i * int(raw_data.info['sfreq'] * 2), 0, 1]
            for i in range(int(raw_data.n_times / (raw_data.info['sfreq'] * 2)))
        ])
    else:
        events, _ = mne.events_from_annotations(raw_data)

    # 2. Filtering
    print("Filtering data (1-50 Hz)...")
    raw_data.filter(l_freq=1, h_freq=50)

    # 3. Detect bad channels
    raw_data.info['bads'] = []  # Customize this logic as needed
    if raw_data.info['bads']:
        print("Interpolating bad channels...")
        raw_data.interpolate_bads()
    else:
        print("No bad channels detected.")

    # 4. ICA for artifact removal
    print("Running ICA...")
    ica = ICA(n_components=20, random_state=97, max_iter=800)
    ica.fit(raw_data)

    eog_channel_name = 'EXG1'
    if eog_channel_name in raw_data.info['ch_names']:
        eog_indices, _ = ica.find_bads_eog(raw_data, ch_name=eog_channel_name)
        ica.exclude = eog_indices
    raw_data_cleaned = ica.apply(raw_data)

    # 5. Epoching (2s fixed-length)
    print("Epoching data...")
    try:
        epochs = mne.Epochs(raw_data_cleaned, events, event_id=None, tmin=0, tmax=2, baseline=(0, 0), detrend=1, preload=True)
        print(f"Epoching completed. {len(epochs)} epochs created.")
    except Exception as e:
        print(f"Error during epoching: {e}")
        return None

    return epochs

# Full subject/session processing pipeline
def process_subject(subject, session, label, bids_root, duration=2.0, overlap=1.0):
    try:
        print(f"🚀 Processing {subject}, {session}")

        bids_path = BIDSPath(
            root=bids_root,
            subject=subject,
            session=session,
            task='rest',
            datatype='eeg',
            suffix='eeg'
        )
        print(f"📂 BIDS path: {bids_path}")

        raw = read_raw_bids(bids_path=bids_path, verbose=True)
        raw.set_channel_types({
            "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
            "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
        })
        raw.set_montage("standard_1020", on_missing="ignore")
        raw.load_data()

        # Filter and ICA steps
        raw.filter(1., 40.)
        raw.notch_filter(60.)
        raw.set_eeg_reference('average')

        ica = ICA(n_components=20, random_state=97)
        ica.fit(raw)

        eog_epochs = create_eog_epochs(raw)
        eog_inds, _ = ica.find_bads_eog(eog_epochs)
        ica.exclude = eog_inds
        raw = ica.apply(raw)

        # Epoching
        epochs = mne.make_fixed_length_epochs(raw, duration=duration, overlap=overlap, preload=True)
        if len(epochs) == 0:
            print(f"⚠️ No epochs for {subject}, session {session}")
            return [], [], []

        # Feature extraction (replace with your actual method)
        print(f"🧠 Extracting features...")
        X_subject, feature_names = extract_band_power_epochs(epochs, compress=False, apply_pca=False)  # No PCA and no compression
        y_subject = [label] * len(X_subject)

        print(f"✅ Features extracted: {len(X_subject)} samples")
        return X_subject, y_subject, feature_names

    except Exception as e:
        print(f"❌ Error processing {subject}, session {session}: {e}")
        return [], [], []

# Dummy entity getter (customize if needed)
def get_entity_vals(bids_root, entity_key='subject'):
    return ['sub-pd1', 'sub-pd2', 'sub-hc1', 'sub-hc2']



## Step 3: Loop Over All Subjects

In [ ]:
import time
import psutil
import numpy as np
import mne
import os
from mne.preprocessing import ICA, create_eog_epochs
from mne_bids import BIDSPath, read_raw_bids
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.decomposition import PCA

def extract_band_power_epochs(
    epochs,
    bands=[(1, 4), (4, 8), (8, 13), (13, 30)],
    compress=False,
    apply_pca=False,
    pca_components=50
):
    psd = epochs.compute_psd()
    psd_data, freqs = psd.get_data(return_freqs=True)
    ch_names = epochs.info['ch_names']
    X = []

    feature_names = []
    for fmin, fmax in bands:
        for ch in ch_names:
            feature_names.append(f"{int(fmin)}-{int(fmax)}Hz_{ch}")

    for trial in psd_data:
        features = []
        for fmin, fmax in bands:
            idx = np.logical_and(freqs >= fmin, freqs <= fmax)
            band_power = trial[:, idx].mean(axis=1)
            features.extend(band_power)
        X.append(features)

    X = np.array(X)
    pca_model = None

    if apply_pca:
        n_samples, n_features = X.shape
        if pca_components is None or pca_components > min(n_samples, n_features):
            pca_components = min(n_samples, n_features)
            print(f"Auto-adjusting PCA to {pca_components} components")

        print(f"Applying PCA: reducing from {X.shape[1]} → {pca_components} dimensions")
        pca_model = PCA(n_components=pca_components, random_state=42)
        X = pca_model.fit_transform(X)
        feature_names = [f"PCA_{i+1}" for i in range(X.shape[1])]

    return X, feature_names, pca_model

def categorize_session(session_name):
    if 'ses-on' in session_name:
        return 'pd-on'
    elif 'ses-off' in session_name:
        return 'pd-off'
    elif 'ses-hc' in session_name:
        return 'ctl'
    else:
        return 'unknown'

def preprocess_and_extract_features(raw_data):
    print("Starting preprocessing and feature extraction...")

    if len(raw_data.annotations) == 0:
        events = np.array([
            [i * int(raw_data.info['sfreq'] * 2), 0, 1]
            for i in range(int(raw_data.n_times / (raw_data.info['sfreq'] * 2)))
        ])
    else:
        events, _ = mne.events_from_annotations(raw_data)

    print("Filtering data (1-50 Hz)...")
    raw_data.filter(l_freq=1, h_freq=50)

    raw_data.info['bads'] = []
    if raw_data.info['bads']:
        print("Interpolating bad channels...")
        raw_data.interpolate_bads()
    else:
        print("No bad channels detected.")

    print("Running ICA...")
    ica = ICA(n_components=20, random_state=97, max_iter=800)
    ica.fit(raw_data)

    eog_channel_name = 'EXG1'
    if eog_channel_name in raw_data.info['ch_names']:
        eog_indices, _ = ica.find_bads_eog(raw_data, ch_name=eog_channel_name)
        ica.exclude = eog_indices
    raw_data_cleaned = ica.apply(raw_data)

    print("Epoching data...")
    try:
        epochs = mne.Epochs(raw_data_cleaned, events, event_id=None, tmin=0, tmax=2, baseline=(0, 0), detrend=1, preload=True)
        print(f"Epoching completed. {len(epochs)} epochs created.")
    except Exception as e:
        print(f"Error during epoching: {e}")
        return None

    return epochs

def process_bids_dataset(dataset_dir):
    valid_subjects = {'pd-on': [], 'pd-off': [], 'ctl': []}
    sample_counts = {'pd-on': 0, 'pd-off': 0, 'ctl': 0}
    feature_counts = {'pd-on': 0, 'pd-off': 0, 'ctl': 0}
    errors = []
    total_samples = 0
    total_features = 0

    X = []
    y = []

    for index, subject_id in enumerate(os.listdir(dataset_dir)):
        subject_path = os.path.join(dataset_dir, subject_id)
        if os.path.isdir(subject_path):
            session_processed = set()
            session_dirs = ['ses-hc', 'ses-on', 'ses-off']
            for session_dir in session_dirs:
                if session_dir in session_processed:
                    continue

                session_path = os.path.join(subject_path, session_dir, 'eeg')
                if os.path.isdir(session_path):
                    bdf_files = [f for f in os.listdir(session_path) if f.endswith('.bdf')]
                    if len(bdf_files) == 0:
                        print(f"No BDF files found for subject {subject_id} in session {session_dir}. Skipping...")
                        continue

                    print(f"Processing subject {subject_id} in session {session_dir} with {len(bdf_files)} BDF files...")

                    try:
                        start_time = time.time()

                        raw_data = mne.io.read_raw_bdf(os.path.join(session_path, bdf_files[0]), preload=True)
                        print(f"Loaded raw data for subject {subject_id}, session {session_dir}.")

                        epochs = preprocess_and_extract_features(raw_data)

                        if epochs is None:
                            raise ValueError(f"Error processing data for subject {subject_id}, session {session_dir}. Skipping.")

                        X_subject, feature_names, _ = extract_band_power_epochs(epochs, compress=False, apply_pca=False)

                        n_samples = len(X_subject)
                        n_features = X_subject.shape[1]
                        total_samples += n_samples
                        total_features = n_features

                        X.extend(X_subject)
                        y.extend([categorize_session(session_dir)] * n_samples)

                        category = categorize_session(session_dir)
                        valid_subjects[category].append(f"{subject_id} ({session_dir})")
                        sample_counts[category] += n_samples
                        feature_counts[category] = n_features

                        session_processed.add(session_dir)
                        print(f"Time taken for subject {subject_id}, session {session_dir}: {time.time() - start_time} seconds")

                    except Exception as e:
                        print(f"Error processing subject {subject_id}, session {session_dir}: {e}")
                        errors.append((subject_id, session_dir, str(e)))

    X = np.array(X)
    y = np.array([0 if label == 'ctl' else 1 if label == 'pd-on' else 2 for label in y])

    print("\nSummary of valid subjects and errors:\n")
    for category in ['pd-on', 'pd-off', 'ctl']:
        print(f"Total number of subjects in {category.upper()}: {len(valid_subjects[category])}")
        print(f"Total samples in {category.upper()}: {sample_counts[category]}")
        print(f"Total features in {category.upper()}: {feature_counts[category]}")
        for subject in valid_subjects[category]:
            print(f"- {subject}")

    print("\nErrors encountered:")
    for subject, session, error in errors:
        print(f"- {subject} ({session}): {error}")

    print(f"\nTotal samples across all categories: {total_samples}")
    print(f"Total features (channels): {total_features}")
    print(f"✅ Feature matrix shape: {X.shape}")
    print(f"✅ Label vector shape: {y.shape}")

    unique_labels, label_counts = np.unique(y, return_counts=True)
    print(f"Label distribution: {dict(zip(unique_labels, label_counts))}")

    return X, y, valid_subjects, errors, total_samples, total_features, sample_counts, feature_counts

# Example usage
dataset_dir = 'PD1'
X, y, valid_subjects, errors, total_samples, total_features, sample_counts, feature_counts = process_bids_dataset(dataset_dir)


### Debug

## Step 4: Train ML Classifier

### Train Random Forest Classifier

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X and y have been processed and are available

# Split data into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Print the shapes of the resulting datasets
print(f"✅ Shape of X_train: {X_train.shape}")
print(f"✅ Shape of X_test: {X_test.shape}")
print(f"✅ Shape of y_train: {y_train.shape}")
print(f"✅ Shape of y_test: {y_test.shape}")

# Check the first epoch in the dataset
print(f"First sample (epoch) shape: {X[0].shape}")
print(f"First sample (epoch) data: {X[0]}")



In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 Random Forest Model
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")


# ***Decision Tree***

In [ ]:
!pip install plotly


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 Decision Tree Model
    model = DecisionTreeClassifier(
        max_depth=20,
        class_weight='balanced',
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")



In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# === Plot the Final Decision Tree Model with Adjustments ===
plt.figure(figsize=(150, 75))  # Decrease the size of the figure to avoid overcrowding

# Visualizing the decision tree using plot_tree function
plot_tree(
    model,  # The trained decision tree model
    filled=True,  # Color the nodes to make them more readable
    feature_names=['Feature' + str(i) for i in range(X_train_scaled.shape[1])],  # Feature names (e.g., Feature0, Feature1, ...)
    class_names=class_names,  # Class names for each target class (Control, PD-On, PD-Off)
    rounded=True,  # Round the corners of the nodes for better aesthetics
    fontsize=10,  # Reduce font size to avoid overcrowding
    proportion=True,  # Show proportion of samples in each node
    precision=2  # Precision of float values displayed on the tree
)

# Add a title to the plot
plt.title("Final Decision Tree Visualization", fontsize=8)

# Adjust layout to avoid clipping of labels and titles
plt.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.05)

# Display the plot
plt.show()



### Step 4.2 : Train Support Vector Machine Classifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 SVM Model (with probability estimation for multi-class)
    model = SVC(
        kernel='linear',  # Using linear kernel for simplicity, can be adjusted
        C=1,  # Regularization parameter
        class_weight='balanced',  # Handle class imbalance
        random_state=42,
        probability=True,  # Enable probability estimates
        verbose=False  # Disable verbose output
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")


## Step 4.3: Train Logistic Regression Classifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 Logistic Regression Model
    model = LogisticRegression(
        max_iter=1000,  # Ensure convergence
        class_weight='balanced',  # Handle class imbalance
        random_state=42,
        multi_class='ovr',  # One-vs-rest for multi-class classification
        solver='liblinear'  # Using liblinear solver for small datasets
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")


## **Transformer Model**

In [ ]:
!pip install torch torchvision

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

**BLOCK 1: Preprocessing and DataLoader**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Entire dataset (X and y)
X = np.concatenate([X_train, X_test], axis=0)  # shape (4497, 160)
y = np.concatenate([y_train, y_test], axis=0)  # shape (4497,)

X = X.astype(np.float32)
y = y.astype(np.int64)

# K-Fold cross-validation will split this
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store all folds' loaders, weights, scalers
fold_data = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"📂 Preparing Fold {fold_idx}...")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Standardize using only training set
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_val_std = scaler.transform(X_val)

    # Convert to PyTorch tensors
    X_train_tensor = torch.tensor(X_train_std, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_std, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)

    # Compute class weights for this fold
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    # Create DataLoaders
    batch_size = 64
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size)

    fold_data.append({
        'train_loader': train_loader,
        'val_loader': val_loader,
        'class_weights': class_weights_tensor,
        'scaler': scaler,
        'X_val': X_val_std,
        'y_val': y_val,
    })



**BLOCK 2: Transformer Model Definition**

In [ ]:
import torch
import torch.nn as nn

class EEGTransformer(nn.Module):
    def __init__(self, input_dim=160, num_classes=3, embed_dim=128, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Linear(input_dim, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dropout=dropout,
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x shape: (B, input_dim)
        x = self.embedding(x).unsqueeze(1)  # (B, 1, embed_dim)
        x = self.transformer_encoder(x)     # (B, 1, embed_dim)
        x = x.squeeze(1)                    # (B, embed_dim)
        return self.classifier(x)           # (B, num_classes)


**BLOCK 3: Training Setup**

In [ ]:
from transformers import get_cosine_schedule_with_warmup
import torch.optim as optim
import torch.nn as nn

def initialize_training_components(input_dim, num_classes, class_weights_tensor, train_loader, epochs=40, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize model
    model = EEGTransformer(input_dim=input_dim, num_classes=num_classes).to(device)

    # Loss function with class weights
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    # Cosine LR scheduler with warmup
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),  # 10% warmup by default
        num_training_steps=total_steps
    )

    return model, loss_fn, optimizer, scheduler, device


**BLOCK 4: Training and Evaluation Loop with tqdm**

In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def train_one_epoch(model, loader, optimizer, loss_fn, scheduler, device):
    model.train()
    total_loss = 0
    correct, total = 0, 0
    loop = tqdm(loader, desc="Train", leave=False)
    
    for xb, yb in loop:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct += (preds.argmax(1) == yb).sum().item()
        total += yb.size(0)

        loop.set_postfix(loss=loss.item(), acc=correct / total)

    avg_loss = total_loss / len(loader)
    avg_acc = correct / total
    return avg_loss, avg_acc

def evaluate(model, loader, device, return_preds=False, show_plot=True):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()

            y_prob.extend(probs)
            y_pred.extend(preds)
            y_true.extend(yb.numpy())

    if return_preds:
        return y_true, y_pred, y_prob

    # Report and Confusion Matrix
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["CTL", "PD-On", "PD-Off"], digits=4))

    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])

    plt.figure(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()


**BLOCK 5: Training Loop**

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])
    plt.figure(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

def plot_macro_roc_curve(y_true, y_prob):
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
    y_prob = np.array(y_prob)
    fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"Macro-average ROC curve (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Macro-Averaged ROC Curve")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Your updated training + evaluation loop:
num_epochs = 40
all_y_true, all_y_pred, all_y_prob = [], [], []

for fold_idx, fold in enumerate(fold_data, 1):
    print(f"\n📁 ===== Fold {fold_idx} =====")

    train_loader = fold["train_loader"]
    val_loader = fold["val_loader"]
    class_weights = fold["class_weights"]

    model, loss_fn, optimizer, scheduler, device = initialize_training_components(
        input_dim=160,
        num_classes=3,
        class_weights_tensor=class_weights,
        train_loader=train_loader,
        epochs=num_epochs,
        lr=5e-4
    )

    # Train loop
    for epoch in range(1, num_epochs + 1):
        print(f"\n🌟 Epoch {epoch}/{num_epochs} — Fold {fold_idx}")
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, scheduler, device)
        print(f"✅ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # Final evaluation on validation set for this fold
    y_true, y_pred, y_prob = evaluate(model, val_loader, device, return_preds=True, show_plot=False)

    # Print classification report
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["CTL", "PD-On", "PD-Off"], digits=4))

    # Plot confusion matrix
    plot_confusion_matrix(y_true, y_pred)

    # Plot macro-average ROC AUC curve
    plot_macro_roc_curve(y_true, y_prob)

    # Collect all fold data for combined stats later
    all_y_true.extend(y_true)
    all_y_pred.extend(y_pred)
    all_y_prob.extend(y_prob)


**BLOCK 6:Evauluate**

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --- Final Classification Report and Confusion Matrix ---
print("\n📊 Final Cross-Validated Classification Report:")
print(classification_report(
    all_y_true, all_y_pred,
    target_names=["CTL", "PD-On", "PD-Off"],
    digits=4
))

cm = confusion_matrix(all_y_true, all_y_pred)
df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])

plt.figure(figsize=(6, 5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix (All Folds Combined)", fontsize=13)
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


# --- Macro-Averaged ROC Curve ---

# Binarize labels for multiclass ROC
y_true_bin = label_binarize(all_y_true, classes=[0, 1, 2])
y_prob = np.array(all_y_prob)

n_classes = y_true_bin.shape[1]
fpr = dict()
tpr = dict()
roc_auc = dict()

# Compute ROC curve and AUC for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Aggregate all FPR points
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

# Interpolate all TPRs at these points and average them
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

mean_tpr /= n_classes

# Compute macro-average AUC
macro_auc = auc(all_fpr, mean_tpr)

# Plot macro-average ROC
plt.figure(figsize=(7, 6))
plt.plot(all_fpr, mean_tpr, color='navy', lw=2, label=f'Macro-average ROC (AUC = {macro_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('📈 Macro-Averaged ROC Curve (All Folds Combined)', fontsize=13)
plt.legend(loc='lower right')
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

